In [165]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backtesting import Strategy, Backtest
from datetime import datetime, timedelta
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [166]:
df = yf.download("QQQ", period="10y", interval="1d")
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
del df['Volume']
df

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open
0,2016-06-13,100.649445,101.487958,100.519010,100.919633
1,2016-06-14,100.649445,101.012799,99.913416,100.379256
2,2016-06-15,100.360641,101.031453,100.192939,100.891699
3,2016-06-16,100.658775,100.751942,99.223986,99.866848
4,2016-06-17,99.478905,100.590560,99.245364,100.450434
...,...,...,...,...,...
2510,2026-06-08,716.070007,723.030029,713.070007,717.809998
2511,2026-06-09,707.830017,725.659973,686.369995,722.979980
2512,2026-06-10,693.690002,711.280029,692.929993,701.659973
2513,2026-06-11,717.119995,718.369995,695.000000,699.289978


In [167]:
def MACD(data, n1, n2, n3):
    fast = data['Close'].ewm(span=n1).mean()
    slow = data['Close'].ewm(span=n2).mean()
    macd = fast - slow
    sig = macd.ewm(span=n3).mean()
    histo = macd - sig
    return macd, sig, histo

df['MACD'], df['SIG'], df['HISTO'] = MACD(df, 12, 26, 9)
df


Price,Date,Close,High,Low,Open,MACD,SIG,HISTO
0,2016-06-13,100.649445,101.487958,100.519010,100.919633,0.000000,0.000000,0.000000
1,2016-06-14,100.649445,101.012799,99.913416,100.379256,0.000000,0.000000,0.000000
2,2016-06-15,100.360641,101.031453,100.192939,100.891699,-0.008956,-0.003670,-0.005285
3,2016-06-16,100.658775,100.751942,99.223986,99.866848,-0.002046,-0.003120,0.001074
4,2016-06-17,99.478905,100.590560,99.245364,100.450434,-0.045197,-0.015637,-0.029560
...,...,...,...,...,...,...,...,...
2510,2026-06-08,716.070007,723.030029,713.070007,717.809998,15.273297,19.626569,-4.353272
2511,2026-06-09,707.830017,725.659973,686.369995,722.979980,12.745737,18.250403,-5.504666
2512,2026-06-10,693.690002,711.280029,692.929993,701.659973,9.492225,16.498767,-7.006542
2513,2026-06-11,717.119995,718.369995,695.000000,699.289978,8.704063,14.939826,-6.235763


In [168]:
def signal(data):
    signal = [0] * len(data)
    for i in range(2,len(data)):
        if (data['MACD'].iloc[i-1] < data['SIG'].iloc[i-1]) and (data['MACD'].iloc[i] > data['SIG'].iloc[i]):
            signal[i] = 1
        elif (data['MACD'].iloc[i-1] > data['SIG'].iloc[i-1]) and (data['MACD'].iloc[i] < data['SIG'].iloc[i]):
            signal[i] = 2
        else:
            signal[i] = 0
        data["signal"] = signal
        
signal(df)
df

Price,Date,Close,High,Low,Open,MACD,SIG,HISTO,signal
0,2016-06-13,100.649445,101.487958,100.519010,100.919633,0.000000,0.000000,0.000000,0
1,2016-06-14,100.649445,101.012799,99.913416,100.379256,0.000000,0.000000,0.000000,0
2,2016-06-15,100.360641,101.031453,100.192939,100.891699,-0.008956,-0.003670,-0.005285,0
3,2016-06-16,100.658775,100.751942,99.223986,99.866848,-0.002046,-0.003120,0.001074,1
4,2016-06-17,99.478905,100.590560,99.245364,100.450434,-0.045197,-0.015637,-0.029560,2
...,...,...,...,...,...,...,...,...,...
2510,2026-06-08,716.070007,723.030029,713.070007,717.809998,15.273297,19.626569,-4.353272,0
2511,2026-06-09,707.830017,725.659973,686.369995,722.979980,12.745737,18.250403,-5.504666,0
2512,2026-06-10,693.690002,711.280029,692.929993,701.659973,9.492225,16.498767,-7.006542,0
2513,2026-06-11,717.119995,718.369995,695.000000,699.289978,8.704063,14.939826,-6.235763,0


In [169]:
def long_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['long_entries'] = df.apply(lambda x: long_entries(x), axis=1)

def short_entries(x):
    offset = 0.002
    if x['signal']==2:
        return x['High'] * (1+offset)
    else:
        return np.nan

df['short_entries'] = df.apply(lambda x: short_entries(x), axis=1)


print(df['signal'].value_counts())
df.shape

signal
0    2311
1     102
2     102
Name: count, dtype: int64


(2515, 11)

In [170]:
df.set_index('Date', inplace=True)

In [171]:
bar = 2200
df1 = df[bar:bar+350].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    row_heights=[0.68,0.32], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x = df1.index, 
                            open = df1['Open'],
                            high = df1['High'],
                            low = df1['Low'],
                            close = df1['Close'],
                            increasing_line_color = 'rgba(19,156,19,0.8)',
                            decreasing_line_color = 'rgba(175,07,49,0.8)',
                            name = 'QQQ'),
                            row=1, col=1)

fig.add_scatter(x=df1.index, y=df1['long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="White"),
                name="Long Entries")

fig.add_scatter(x=df1.index, y=df1['short_entries'], mode="markers",
                marker=dict(size=7, symbol='cross', color="gold"),
                name="Short Entries")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.MACD, 
                         line=dict(color='red', width=1),
                         name='MACD'),
                         row=2, col=1)

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.SIG, 
                         line=dict(color='lightseagreen', width=1),
                         name='SIGNAL'),
                         row=2, col=1)

fig.add_trace(go.Bar(x=df1.index, 
                     y=df1.HISTO, 
                     name="Histogram",
                     marker=dict(color='gray')),
                     row=2, col=1)

fig.update_layout(autosize=False, width=1100, height=700, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.show()

In [172]:
def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99)
        
        elif self.signal==2:
            if self.position.is_long or not self.position:
                self.position.close()
                self.sell(size=0.99)
                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Backtest.run:   0%|          | 0/2514 [00:00<?, ?bar/s]

Start                     2016-06-13 00:00:00
End                       2026-06-12 00:00:00
Duration                   3651 days 00:00:00
Exposure Time [%]                    99.64215
Equity Final [$]                  45942.60101
Equity Peak [$]                  108843.55766
Commissions [$]                    15499.6881
Return [%]                           -54.0574
Buy & Hold Return [%]               616.68555
Return (Ann.) [%]                     -7.4973
Volatility (Ann.) [%]                19.41024
CAGR [%]                             -5.22684
Sharpe Ratio                         -0.38625
Sortino Ratio                        -0.50502
Calmar Ratio                         -0.11691
Alpha [%]                            73.59087
Beta                                 -0.20699
Max. Drawdown [%]                   -64.12996
Avg. Drawdown [%]                   -10.69961
Max. Drawdown Duration     2223 days 00:00:00
Avg. Drawdown Duration      364 days 00:00:00
# Trades                          

In [173]:
df = yf.download("QQQ", period="10y", interval="1d")
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
del df['Volume']
df

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open
0,2016-06-13,100.649445,101.487958,100.519010,100.919633
1,2016-06-14,100.649445,101.012799,99.913416,100.379256
2,2016-06-15,100.360649,101.031460,100.192946,100.891707
3,2016-06-16,100.658791,100.751957,99.224001,99.866863
4,2016-06-17,99.478912,100.590567,99.245372,100.450442
...,...,...,...,...,...
2510,2026-06-08,716.070007,723.030029,713.070007,717.809998
2511,2026-06-09,707.830017,725.659973,686.369995,722.979980
2512,2026-06-10,693.690002,711.280029,692.929993,701.659973
2513,2026-06-11,717.119995,718.369995,695.000000,699.289978


In [174]:
def MACD(data, n1, n2, n3):
    fast = data['Close'].rolling(n1).mean()
    slow = data['Close'].rolling(n2).mean()
    macd = fast - slow
    sig = macd.rolling(n3).mean()
    histo1 = macd - sig
    histo = histo1.rolling(3).mean()
    return macd, sig, histo

df['MACD'], df['SIG'], df['HISTO'] = MACD(df, 12, 26, 9)
df

Price,Date,Close,High,Low,Open,MACD,SIG,HISTO
0,2016-06-13,100.649445,101.487958,100.519010,100.919633,NaN,NaN,NaN
1,2016-06-14,100.649445,101.012799,99.913416,100.379256,NaN,NaN,NaN
2,2016-06-15,100.360649,101.031460,100.192946,100.891707,NaN,NaN,NaN
3,2016-06-16,100.658791,100.751957,99.224001,99.866863,NaN,NaN,NaN
4,2016-06-17,99.478912,100.590567,99.245372,100.450442,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2510,2026-06-08,716.070007,723.030029,713.070007,717.809998,15.597305,19.522645,-3.391152
2511,2026-06-09,707.830017,725.659973,686.369995,722.979980,13.745255,18.446163,-3.997105
2512,2026-06-10,693.690002,711.280029,692.929993,701.659973,10.957372,17.243384,-4.970753
2513,2026-06-11,717.119995,718.369995,695.000000,699.289978,8.494933,15.834872,-6.108953


In [175]:
def signal(data):
    signal = [0] * len(data)
    for i in range(2,len(data)):
        if (data['HISTO'].iloc[i-1] < data['HISTO'].iloc[i-2]) and (data['HISTO'].iloc[i] > data['HISTO'].iloc[i-1]) and (data['HISTO'].iloc[i] < 0):
            signal[i] = 1
        elif (data['MACD'].iloc[i-1] > data['SIG'].iloc[i-1]) and (data['MACD'].iloc[i] < data['SIG'].iloc[i]):
            signal[i] = 2
        else:
            signal[i] = 0
        data["signal"] = signal
        
signal(df)
df

Price,Date,Close,High,Low,Open,MACD,SIG,HISTO,signal
0,2016-06-13,100.649445,101.487958,100.519010,100.919633,NaN,NaN,NaN,0
1,2016-06-14,100.649445,101.012799,99.913416,100.379256,NaN,NaN,NaN,0
2,2016-06-15,100.360649,101.031460,100.192946,100.891707,NaN,NaN,NaN,0
3,2016-06-16,100.658791,100.751957,99.224001,99.866863,NaN,NaN,NaN,0
4,2016-06-17,99.478912,100.590567,99.245372,100.450442,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...
2510,2026-06-08,716.070007,723.030029,713.070007,717.809998,15.597305,19.522645,-3.391152,0
2511,2026-06-09,707.830017,725.659973,686.369995,722.979980,13.745255,18.446163,-3.997105,0
2512,2026-06-10,693.690002,711.280029,692.929993,701.659973,10.957372,17.243384,-4.970753,0
2513,2026-06-11,717.119995,718.369995,695.000000,699.289978,8.494933,15.834872,-6.108953,0


In [176]:
def long_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['long_entries'] = df.apply(lambda x: long_entries(x), axis=1)

def short_entries(x):
    offset = 0.002
    if x['signal']==2:
        return x['High'] * (1+offset)
    else:
        return np.nan

df['short_entries'] = df.apply(lambda x: short_entries(x), axis=1)


print(df['signal'].value_counts())
df.shape

signal
0    2306
1     119
2      90
Name: count, dtype: int64


(2515, 11)

In [177]:
df.set_index('Date', inplace=True)

In [178]:
bar = 2200
df1 = df[bar:bar+350].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    row_heights=[0.68,0.32], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x = df1.index, 
                            open = df1['Open'],
                            high = df1['High'],
                            low = df1['Low'],
                            close = df1['Close'],
                            increasing_line_color = 'rgba(19,156,19,0.8)',
                            decreasing_line_color = 'rgba(175,07,49,0.8)',
                            name = 'QQQ'),
                            row=1, col=1)

fig.add_scatter(x=df1.index, y=df1['long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="White"),
                name="Long Entries")

fig.add_scatter(x=df1.index, y=df1['short_entries'], mode="markers",
                marker=dict(size=7, symbol='cross', color="gold"),
                name="Short Entries")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.MACD, 
                         line=dict(color='red', width=1),
                         name='MACD'),
                         row=2, col=1)

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.SIG, 
                         line=dict(color='lightseagreen', width=1),
                         name='SIGNAL'),
                         row=2, col=1)

fig.add_trace(go.Bar(x=df1.index, 
                     y=df1.HISTO, 
                     name="Histogram",
                     marker=dict(color='white')),
                     row=2, col=1)

fig.update_layout(autosize=False, width=1100, height=700, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.show()

In [179]:
def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, tp=1.12*price)
        
        elif self.signal==2:
            if self.position.is_long or not self.position:
                self.position.close()
                #self.sell(size=0.99)
                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Backtest.run:   0%|          | 0/2514 [00:00<?, ?bar/s]

Start                     2016-06-13 00:00:00
End                       2026-06-12 00:00:00
Duration                   3651 days 00:00:00
Exposure Time [%]                    69.98012
Equity Final [$]                 467681.41986
Equity Peak [$]                  483591.00543
Commissions [$]                   18156.58829
Return [%]                          367.68142
Buy & Hold Return [%]               616.68555
Return (Ann.) [%]                    16.71541
Volatility (Ann.) [%]                20.24975
CAGR [%]                             11.23499
Sharpe Ratio                          0.82546
Sortino Ratio                          1.3716
Calmar Ratio                          0.63958
Alpha [%]                            -3.26027
Beta                                  0.60151
Max. Drawdown [%]                   -26.13479
Avg. Drawdown [%]                    -2.49543
Max. Drawdown Duration      569 days 00:00:00
Avg. Drawdown Duration       26 days 00:00:00
# Trades                          

In [180]:
df = yf.download("TSLA", period="10y", interval="1d")
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
del df['Volume']
df

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open
0,2016-06-13,14.524667,15.051333,14.510667,14.633333
1,2016-06-14,14.330667,14.813333,14.168667,14.592000
2,2016-06-15,14.513333,14.793333,14.342000,14.463333
3,2016-06-16,14.528667,14.536000,14.233333,14.494667
4,2016-06-17,14.364667,14.666000,14.300000,14.520667
...,...,...,...,...,...
2510,2026-06-08,408.950012,412.940002,394.720001,396.329987
2511,2026-06-09,396.679993,418.500000,384.239990,411.029999
2512,2026-06-10,381.589996,397.089996,380.149994,391.540009
2513,2026-06-11,399.149994,399.540009,380.660004,388.279999


In [181]:
def MACD(data, n1, n2, n3):
    fast = data['Close'].rolling(n1).mean()
    slow = data['Close'].rolling(n2).mean()
    macd = fast - slow
    sig = macd.rolling(n3).mean()
    histo1 = macd - sig
    histo = histo1.rolling(n3).mean()
    return macd, sig, histo

df['MACD'], df['SIG'], df['HISTO'] = MACD(df, 12, 26, 9)
df

Price,Date,Close,High,Low,Open,MACD,SIG,HISTO
0,2016-06-13,14.524667,15.051333,14.510667,14.633333,NaN,NaN,NaN
1,2016-06-14,14.330667,14.813333,14.168667,14.592000,NaN,NaN,NaN
2,2016-06-15,14.513333,14.793333,14.342000,14.463333,NaN,NaN,NaN
3,2016-06-16,14.528667,14.536000,14.233333,14.494667,NaN,NaN,NaN
4,2016-06-17,14.364667,14.666000,14.300000,14.520667,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2510,2026-06-08,408.950012,412.940002,394.720001,396.329987,3.517951,11.500626,-5.733323
2511,2026-06-09,396.679993,418.500000,384.239990,411.029999,1.528399,9.319188,-6.694426
2512,2026-06-10,381.589996,397.089996,380.149994,391.540009,-1.753269,7.037920,-7.503383
2513,2026-06-11,399.149994,399.540009,380.660004,388.279999,-4.999423,4.580627,-8.251834


In [182]:
def signal(data):
    signal = [0] * len(data)
    for i in range(2,len(data)):
        if (data['HISTO'].iloc[i-1] < data['HISTO'].iloc[i-2]) and (data['HISTO'].iloc[i] > data['HISTO'].iloc[i-1]) and (data['HISTO'].iloc[i] < 0):
            signal[i] = 1
        elif (data['MACD'].iloc[i-1] > data['SIG'].iloc[i-1]) and (data['MACD'].iloc[i] < data['SIG'].iloc[i]):
            signal[i] = 2
        else:
            signal[i] = 0
        data["signal"] = signal
        
signal(df)
df

Price,Date,Close,High,Low,Open,MACD,SIG,HISTO,signal
0,2016-06-13,14.524667,15.051333,14.510667,14.633333,NaN,NaN,NaN,0
1,2016-06-14,14.330667,14.813333,14.168667,14.592000,NaN,NaN,NaN,0
2,2016-06-15,14.513333,14.793333,14.342000,14.463333,NaN,NaN,NaN,0
3,2016-06-16,14.528667,14.536000,14.233333,14.494667,NaN,NaN,NaN,0
4,2016-06-17,14.364667,14.666000,14.300000,14.520667,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...
2510,2026-06-08,408.950012,412.940002,394.720001,396.329987,3.517951,11.500626,-5.733323,0
2511,2026-06-09,396.679993,418.500000,384.239990,411.029999,1.528399,9.319188,-6.694426,0
2512,2026-06-10,381.589996,397.089996,380.149994,391.540009,-1.753269,7.037920,-7.503383,0
2513,2026-06-11,399.149994,399.540009,380.660004,388.279999,-4.999423,4.580627,-8.251834,0


In [183]:
def long_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['long_entries'] = df.apply(lambda x: long_entries(x), axis=1)

def short_entries(x):
    offset = 0.002
    if x['signal']==2:
        return x['High'] * (1+offset)
    else:
        return np.nan

df['short_entries'] = df.apply(lambda x: short_entries(x), axis=1)


print(df['signal'].value_counts())
df.shape

signal
0    2350
2      83
1      82
Name: count, dtype: int64


(2515, 11)

In [184]:
df.set_index('Date', inplace=True)

In [185]:
bar = 2200
df1 = df[bar:bar+350].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    row_heights=[0.68,0.32], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x = df1.index, 
                            open = df1['Open'],
                            high = df1['High'],
                            low = df1['Low'],
                            close = df1['Close'],
                            increasing_line_color = 'rgba(19,156,19,0.8)',
                            decreasing_line_color = 'rgba(175,07,49,0.8)',
                            name = 'QQQ'),
                            row=1, col=1)

fig.add_scatter(x=df1.index, y=df1['long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="White"),
                name="Long Entries")

fig.add_scatter(x=df1.index, y=df1['short_entries'], mode="markers",
                marker=dict(size=7, symbol='cross', color="gold"),
                name="Short Entries")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.MACD, 
                         line=dict(color='red', width=1),
                         name='MACD'),
                         row=2, col=1)

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.SIG, 
                         line=dict(color='lightseagreen', width=1),
                         name='SIGNAL'),
                         row=2, col=1)

fig.add_trace(go.Bar(x=df1.index, 
                     y=df1.HISTO, 
                     name="Histogram",
                     marker=dict(color='white')),
                     row=2, col=1)

fig.update_layout(autosize=False, width=1100, height=700, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.show()

In [188]:
def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99)
        
        elif self.signal==2:
            if self.position.is_long or not self.position:
                self.position.close()
                self.sell(size=0.99, sl=1.06*price, tp=0.9*price)
                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Backtest.run:   0%|          | 0/2514 [00:00<?, ?bar/s]

Start                     2016-06-13 00:00:00
End                       2026-06-12 00:00:00
Duration                   3651 days 00:00:00
Exposure Time [%]                    82.70378
Equity Final [$]               22633260.36213
Equity Peak [$]                22633260.36213
Commissions [$]                  649233.79893
Return [%]                        22533.26036
Buy & Hold Return [%]              2698.20528
Return (Ann.) [%]                     72.1642
Volatility (Ann.) [%]                93.75531
CAGR [%]                             45.38841
Sharpe Ratio                          0.76971
Sortino Ratio                         2.23437
Calmar Ratio                          1.48079
Alpha [%]                          21118.9506
Beta                                  0.52417
Max. Drawdown [%]                   -48.73361
Avg. Drawdown [%]                    -9.38487
Max. Drawdown Duration      448 days 00:00:00
Avg. Drawdown Duration       41 days 00:00:00
# Trades                          

In [189]:
trades = stats['_trades']
trades['CumulativePnL'] = trades['PnL'].cumsum()

fig_trades = go.Figure()

fig_trades.add_trace(go.Scatter(x=trades['EntryTime'], 
                                      y=trades['CumulativePnL'], 
                                      mode='lines', 
                                      name='Cumulative PnL', 
                                      line=dict(color='#00df9a')))

fig_trades.update_layout(title='Strategy PnL',
                         template="plotly_dark",
                         autosize=False,
                         width=1100,
                         height=700,
                        )

fig_trades.update_yaxes(gridcolor="#171717")
fig_trades.update_xaxes(gridcolor="#171717")

fig_trades.write_image("PNL.png", scale=2)

fig_trades.show()